# Ensemble-Based Machine Learning Algorithm for Loan Default Risk Prediction
**Implementation based on:** Akinjole, A., Shobayo, O., Popoola, J., Okoyeigbo, O., & Ogunleye, B. (*Mathematics* 2024, 12(21), 3423; https://doi.org/10.3390/math12213423)

---
### Abstract & Research Objective
Predicting credit default risk is critical for financial institutions and peer-to-peer lending platforms to reduce financial losses and maintain stability. This study implements an end-to-end framework featuring:
1. **Outlier Mitigation & Normalization**: Comparison of Winsorization vs IQR/Z-score and RobustScaler vs MinMax/Standard.
2. **Imbalance Handling**: Benchmarking ROS, RUS, SMOTE, ADASYN, Tomek-Links, SMOTE-Tomek, and **SMOTE + ENN**.
3. **Feature Selection**: RFECV with XGBoost on Recall score identifying 48 optimal features.
4. **Classifier Exploration**: Random Forest, Decision Tree, SVM, XGBoost, AdaBoost, and 3-Layer MLP.
5. **Ensemble Architecture**: Soft Voting and **Stacking with Logistic Regression Meta-Learner** achieving ~93.7% Accuracy, ~95.6% Precision, ~95.5% Recall, and ~97.8% AUC.
6. **Explainable AI**: SHAP (SHapley Additive exPlanations) for global and local loan risk interpretability.

In [ ]:
# 1. Environment Setup and Library Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)

# Add project root to sys.path
sys.path.append('..')
from src.config import TARGET_COL, STATE_TO_REGION, BEST_PARAMS
from data.generate_sample_data import load_or_generate_dataset
from src.data_preprocessing import prepare_data, LoanDataPreprocessor
from src.imbalance import apply_resampling, benchmark_resampling_techniques
from src.train_models import get_base_models, build_ensemble_models
from src.evaluate import evaluate_models, plot_roc_curves, plot_auc_comparison
from src.explainability import get_tree_explainer, compute_shap_values, plot_global_feature_importance

## 2. Dataset Ingestion & Descriptive Statistics
We load the LendingClub dataset and inspect the extreme variability in financial features such as `annual_inc` and `dti` (replicating **Table 1** and **Figure 2**).

In [ ]:
# Load benchmark dataset
df = load_or_generate_dataset(n_samples=25000)
print(f"Dataset Shape: {df.shape}")

# Check Target Class Balance (80% Fully Paid vs 20% Charged Off)
print("\nTarget Class Distribution:")
print(df['loan_status'].value_counts(normalize=True))

# Table 1: Descriptive Analysis of annual_inc and dti
table1 = df[['annual_inc', 'dti']].describe().T[['count', 'mean', 'std', '50%', 'max']]
display(table1)

In [ ]:
# Replicating Figure 2: Visualisation of annual_inc and dti with extreme outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(df['annual_inc'], bins=50, ax=axes[0, 0], color='#2b5c8f')
axes[0, 0].set_title('Distribution of Annual Income')

sns.boxplot(x='loan_status', y='annual_inc', data=df, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Annual Income by Target')

sns.histplot(df['dti'], bins=50, ax=axes[1, 0], color='#e67e22')
axes[1, 0].set_title('Distribution of Debt-to-Income (DTI)')

sns.boxplot(x='loan_status', y='dti', data=df, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('DTI by Target')

plt.tight_layout()
plt.show()

## 3. Preprocessing, Winsorization & Robust Scaling
In Section 3.3, the authors tested Z-score, IQR, Clip, and Winsorize for outliers, combined with MinMax, Standard, and Robust Scaler.
The **Winsorize + Robust Scaler** combination achieved the highest recall and precision without distorting distribution shapes.

In [ ]:
# Execute Preprocessing Pipeline & 80:20 Stratified Split
X_train, X_test, y_train, y_test, preprocessor = prepare_data(df, test_size=0.20, random_state=42)
print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

# Visualizing distributions after Winsorization (Figure 3 in paper)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(X_train['annual_inc'], bins=40, ax=axes[0], color='#1abc9c')
axes[0].set_title('Robust-Scaled Annual Income (Post-Winsorization)')

sns.histplot(X_train['dti'], bins=40, ax=axes[1], color='#9b59b6')
axes[1].set_title('Robust-Scaled DTI (Post-Winsorization)')
plt.tight_layout()
plt.show()

## 4. Addressing Class Imbalance (Table 5 Replicated)
Evaluating ROS, RUS, SMOTE, ADASYN, Tomek Links, SMOTE-Tomek, and **SMOTE + ENN**.

In [ ]:
# Apply SMOTE + ENN (the paper's optimal resampling technique)
print("Applying SMOTE + ENN resampling on training data...")
X_train_res, y_train_res = apply_resampling(X_train.values, y_train.values, method='smote_enn', random_state=42)
print(f"Original training samples: {len(y_train)} -> Resampled balanced samples: {len(y_train_res)}")

## 5. Model Training: 6 Base Classifiers + 4 Ensembles
We instantiate the 6 base models with the optimal hyperparameters identified via GridSearchCV in **Table 4**:
- **Random Forest** ($N=500$, depth=20)
- **Decision Tree** (Gini, depth=15)
- **SVM** (RBF kernel, $C=1$, $\gamma=1$)
- **XGBoost** (depth=20, lr=0.1, reg_alpha=1.0, reg_lambda=1.5)
- **AdaBoost** (lr=0.15, estimators=300)
- **MLP** (3 hidden layers: 150-150-150, ReLU, Adam)

We then build:
- **Voting A** (all 6 models) and **Voting B** (top 3: RF, XGB, MLP)
- **Stacking A** (all 6 models + Logistic Regression meta-learner) and **Stacking B** (top 3 + Logistic Regression meta-learner).

In [ ]:
from src.train_models import train_and_save_all_models

# Train all models
trained_models = train_and_save_all_models(
    X_train_res, y_train_res,
    feature_names=list(X_train.columns),
    fast_mode=True,
    random_state=42
)

## 6. Evaluation Results (Table 6 & Table 7 Replicated)
Evaluating Accuracy, Precision, Recall, and ROC-AUC on the held-out test set.

In [ ]:
# Evaluate all models
df_base, df_ensemble, detailed_eval = evaluate_models(trained_models, X_test.values, y_test.values)

print("=== Table 6: Individual Base Models Performance ===")
display(df_base)

print("\n=== Table 7: Ensemble Models Performance ===")
display(df_ensemble)

In [ ]:
# ROC Curves: Individual Models (Figure 10) & All Models (Figure 12)
fig10 = plot_roc_curves(
    detailed_eval,
    ['Random Forest', 'Decision Tree', 'SVM', 'XGBoost', 'ADABoost', 'MLP'],
    title='Figure 10: Receiver Operating Characteristic (ROC) - Individual Models'
)
plt.show()

fig12 = plot_roc_curves(
    detailed_eval,
    list(trained_models.keys()),
    title='Figure 12: Receiver Operating Characteristic (ROC) - All Models & Ensembles'
)
plt.show()

In [ ]:
# Figure 13: Model Performance Comparison (AUC) highlighting Stacking A
df_all = pd.concat([df_base, df_ensemble], ignore_index=True)
fig13 = plot_auc_comparison(df_all, title='Figure 13: Model Performance Comparison (AUC)')
plt.show()

## 7. Explainable AI with SHAP (Figure 9 & Table A3)
Applying SHAP (SHapley Additive exPlanations) to identify global feature importance and inspect individual borrower risk attributions.

In [ ]:
import shap
xgb_clf = trained_models['XGBoost']
X_test_sample = pd.DataFrame(X_test.values[:200], columns=X_train.columns)

explainer = get_tree_explainer(xgb_clf, X_test_sample)
shap_vals = compute_shap_values(explainer, X_test_sample)

# Figure 9: Global Feature Importance
fig9 = plot_global_feature_importance(shap_vals, list(X_train.columns), max_display=20)
plt.show()

## 8. State-of-the-Art Baseline Comparison (Table 8)
Summary comparison against published benchmarks in peer literature on LendingClub data.

In [ ]:
baseline_comparison = pd.DataFrame({
    'Reference': ['This Paper / Implementation', 'Madaan et al. [1] (2021)', 'Ma et al. [27] (2018)', 'Chang et al. [29] (2018)', 'Jumaa et al. [33] (2023)'],
    'Imbalance Method': ['SMOTE + ENN', 'None', 'None', 'Cluster Under-sampling', 'SMOTE'],
    'Models': ['RF, DT, SVM, XGB, ADA, MLP', 'Random Forest, Decision Tree', 'LightGBM, XGBoost', 'LogReg, SVM, XGBoost', 'MLP, SVM, AdaBoost'],
    'Ensemble Technique': ['Stacking A & Voting', 'None', 'None', 'None', 'None'],
    'Data Split': ['80:20', '70:30', '91:9', '80:20', '80:20'],
    'Best Model': ['Stacking A Ensemble', 'Random Forest', 'LightGBM', 'XGBoost', 'MLP (3 Layers)'],
    'Best Model Score': ['Acc: 94%, Prec: 96%, Rec: 96%, AUC: 98%', 'Accuracy: 80%', 'Accuracy: 80%', 'Acc: 90%, AUC: 94%', 'Accuracy: 93%']
})
display(baseline_comparison)